In [ ]:
# Colab dependency setup.
# Run this first in Google Colab, or use Runtime > Run all.
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    get_ipython().system(f"{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib openai")


# Notebook 2 — Trajectory Evaluation
### Evaluating AI Agents: From Component Checks to Adversarial Defense

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and inspect pre-recorded agent traces.
2. Run reusable **trajectory assertions** across a trace suite.
3. Detect loops, duplicate tool calls, excessive steps, missing recovery, and cost/latency violations.
4. Produce a compact **per-trace pass/fail report** for regression monitoring.

> **Why trajectory evaluation?**  
> Two agents can produce the same final answer by very different paths.  
> One path may be efficient and recoverable. The other may be wasteful, stuck in loops, or silently failing.  
> Trajectory evaluation makes path quality measurable — not just output quality.

In [ ]:
#@title Setup and runtime options
import os
import sys
from pathlib import Path

#@markdown Check `USE_OPENAI` to call a small baseline OpenAI model. Add `OPENAI_API_KEY` in Colab Secrets first.
USE_OPENAI = False #@param {type:"boolean"}
OPENAI_MODEL = "gpt-4.1-nano" #@param {type:"string"}

REPO_URL = "https://github.com/AmmarMohanna/packt-ai-agents-eval.git"
REPO_NAME = "packt-ai-agents-eval"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_PATH = Path("/content") / REPO_NAME
    os.chdir("/content")
    get_ipython().system(f"rm -rf {REPO_PATH}")
    get_ipython().system(f"git clone -q {REPO_URL} {REPO_PATH}")
    PROJECT_ROOT = REPO_PATH
    if not (PROJECT_ROOT / "notebooks").is_dir():
        raise RuntimeError(f"Could not clone course repo into {PROJECT_ROOT}. Re-run this cell or restart the runtime.")
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    PROJECT_ROOT = next(
        (path for path in candidates if (path / "src").is_dir() and (path / "data").is_dir()),
        cwd,
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT / "notebooks")

import importlib
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)

OPENAI_API_KEY = None
if USE_OPENAI:
    if not IN_COLAB:
        raise RuntimeError("OpenAI mode is configured for Google Colab Secrets.")
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise RuntimeError("Add OPENAI_API_KEY in the Colab Secrets panel, then rerun the notebook.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

from src.trace_utils import load_jsonl, trace_to_dataframe, summarize_trace
from src.openai_agents import generate_trace_with_openai
from src.metrics import (
    count_steps, total_latency, total_tokens,
    has_duplicate_consecutive_tool_calls, has_loop,
    has_recovery_after_failure, run_trajectory_assertions,
)

pd.set_option("display.max_colwidth", 80)
print("Setup complete.")

In [ ]:
# ── Load or generate traces ───────────────────────────────────────────────
DATA_PATH = os.path.join("..", "data", "trajectories.jsonl")
traces = load_jsonl(DATA_PATH)

if USE_OPENAI:
    print(f"Generating OpenAI trajectories with {OPENAI_MODEL}...")
    traces = [generate_trace_with_openai(t, api_key=OPENAI_API_KEY, model=OPENAI_MODEL) for t in traces]
else:
    print("Using pre-recorded deterministic traces.")

print(f"Loaded {len(traces)} agent traces.")
print("\nTrace IDs and profiles:")
for t in traces:
    print(f"  {t['trace_id']:10s}  profile={t.get('agent_profile','?'):25s}  steps={len(t['steps'])}")


In [ ]:
# ── Inspect one clean trace and one problematic trace ─────────────────────
def print_trace(trace, label=""):
    print(f"\n{'─'*60}")
    print(f"  {label}  |  trace_id={trace['trace_id']}  |  profile={trace.get('agent_profile')}")
    print(f"  task: {trace['task'][:65]}")
    print(f"{'─'*60}")
    for s in trace["steps"]:
        status_icon = "✓" if s["status"] == "success" else "✗"
        args_str = str(s.get("arguments", {}))[:45]
        print(f"  Step {s['step_id']} {status_icon}  {s.get('tool_name','—'):22s}  args={args_str}")
        print(f"         latency={s['latency_ms']}ms  tokens={s['tokens']}")
    print(f"  Final: {trace['final_answer'][:70]}")

if USE_OPENAI:
    clean = traces[0]
    problem = traces[min(1, len(traces) - 1)]
else:
    clean = next(t for t in traces if t.get("agent_profile") == "clean")
    problem = next(t for t in traces if t.get("agent_profile") == "loop")

clean_label = "OPENAI TRACE 1" if USE_OPENAI else "CLEAN"
problem_label = "OPENAI TRACE 2" if USE_OPENAI else "PROBLEMATIC (loop)"
print_trace(clean, label=clean_label)
print_trace(problem, label=problem_label)

In [ ]:
# ── Build a summary dataframe across all traces ───────────────────────────
summaries = [summarize_trace(t) for t in traces]
df_summary = pd.DataFrame(summaries)

print("Trace-level summary:")
print(df_summary[["trace_id","n_steps","total_latency_ms","total_tokens","n_failures"]].to_string(index=False))

In [ ]:
# ── Run trajectory assertions over all traces ─────────────────────────────
#
# Assertion thresholds — tune these for your production SLAs.
MAX_STEPS      = 6      # more than 6 steps is considered bloated
MAX_LATENCY_MS = 5000   # 5 second budget per trace
MAX_TOKENS     = 1500   # token budget per trace

assertion_results = [
    run_trajectory_assertions(t, max_steps=MAX_STEPS,
                                 max_latency_ms=MAX_LATENCY_MS,
                                 max_tokens=MAX_TOKENS)
    for t in traces
]
df_assert = pd.DataFrame(assertion_results)

# Add profile column for readability
profile_map = {t["trace_id"]: t.get("agent_profile", "?") for t in traces}
df_assert.insert(1, "profile", df_assert["trace_id"].map(profile_map))

print(f"Assertions run with: max_steps={MAX_STEPS}, max_latency={MAX_LATENCY_MS}ms, max_tokens={MAX_TOKENS}")

In [ ]:
# ── Pass/fail table by trace ───────────────────────────────────────────────
assert_cols = [c for c in df_assert.columns if c.startswith("assert_")]

# Count passing assertions per trace
df_assert["pass_count"] = df_assert[assert_cols].sum(axis=1)
df_assert["total_assertions"] = len(assert_cols)
df_assert["all_pass"] = df_assert["pass_count"] == len(assert_cols)

display_cols = ["trace_id", "profile"] + assert_cols + ["pass_count", "all_pass"]

# Rename boolean columns to ✓/✗ for readability
def bool_to_symbol(val):
    return "✓" if val else "✗"

df_display = df_assert[display_cols].copy()
for col in assert_cols + ["all_pass"]:
    df_display[col] = df_display[col].map(bool_to_symbol)

print("Per-trace assertion results (✓ = pass, ✗ = fail):")
print(df_display.to_string(index=False))

In [ ]:
# ── Aggregate failure counts by assertion type ─────────────────────────────
failure_counts = {col: int((~df_assert[col]).sum()) for col in assert_cols}
df_failures = pd.DataFrame([
    {"assertion": k.replace("assert_", ""), "failures": v,
     "failure_rate": f"{v/len(traces):.0%}"}
    for k, v in failure_counts.items()
]).sort_values("failures", ascending=False)

print("Failure summary by assertion type:")
print(df_failures.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(df_failures["assertion"], df_failures["failures"], color="#e06c75")
ax.set_xlabel("Number of traces failing")
ax.set_title("Trajectory Assertion Failures by Type")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "nb02_assertion_failures.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ── Inspect one failed trace in detail ────────────────────────────────────
# Focus on the 'failure_no_recovery' case — agent hits an error and stops.

if USE_OPENAI:
    failed_trace_ids = set(df_assert.loc[~df_assert["all_pass"], "trace_id"])
    failed_trace = next((t for t in traces if t["trace_id"] in failed_trace_ids), traces[0])
else:
    failed_trace = next(t for t in traces if t.get("agent_profile") == "failure_no_recovery")
result = run_trajectory_assertions(failed_trace, max_steps=MAX_STEPS,
                                                  max_latency_ms=MAX_LATENCY_MS,
                                                  max_tokens=MAX_TOKENS)

print_trace(failed_trace, label="FAILED TRACE (no recovery)")
print("\nAssertion results for this trace:")
for k, v in result.items():
    if k.startswith("assert_"):
        status = "✓ PASS" if v else "✗ FAIL"
        print(f"  {k.replace('assert_',''):35s} {status}")

## Interpreting Trajectory Results

### Hard failure vs. warning

Not every assertion violation has the same severity. Use this decision guide:

| Violation | Recommended treatment |
|-----------|----------------------|
| **No recovery after failure** | Hard failure — agent abandons the task silently. Always block. |
| **Loop detected** | Hard failure — agent is stuck and will not self-resolve. Always block. |
| **Duplicate consecutive calls** | Hard failure — wastes compute and signals a planning bug. Block. |
| **Exceeded max steps** | Warning — may be an unusually complex task. Flag for review. |
| **Exceeded latency budget** | Warning — depends on user-facing SLA. Can be soft limit. |
| **Exceeded token budget** | Warning — monitor cost impact. Block only above critical threshold. |

**Key insight:** The agent in `tr_007` (failure_no_recovery) and `tr_020` (failure_no_recovery) both produce a final answer — they look fine if you only evaluate outputs. Trajectory evaluation reveals the silent failure that output scoring misses.

## Try It Yourself

**Extension exercise** — Tighten or relax the assertion thresholds and observe how the report changes:

```python
# Try stricter thresholds:
MAX_STEPS      = 3     # very strict
MAX_LATENCY_MS = 2000  # tight latency SLA
MAX_TOKENS     = 600   # low token budget

# Or more lenient:
MAX_STEPS      = 10
MAX_LATENCY_MS = 10000
MAX_TOKENS     = 5000

# Re-run the assertion block above and see which traces flip from pass to fail.
```

**Questions to explore:**
- Which profiles always fail regardless of thresholds? Which only fail under strict settings?
- Write a new assertion: `assert_has_citation` — checks that at least one step uses the `cite_sources` tool.
- How would you integrate these assertions into a CI/CD pipeline?